# Whisper Batch Transcriber v4 (Colab Pro, full run, session order)

Built for the full run of all videos with **large-v3**. Files are processed in **session-number order** (Session 2 before Session 10), and you can limit a run to a range of sessions. Output goes to `transcripts_large_v3/`.

**Before running**
1. Runtime -> Change runtime type -> choose the best GPU your plan offers (**A100 or L4** if listed, otherwise T4). Save.
2. Hugging Face token in Colab Secrets (key icon on the left): name `HF_TOKEN`, Notebook access ON. (Optional; removes the warning.)
3. All mp3 files are in `MyDrive/transcribe/audio/`.

**Run order:** Step 1 -> Step 1b -> Step 2 -> Step 3. If the session disconnects, run the same four cells again; finished files are skipped.

**Per video you get three files:** `.txt` (timestamped lines), `.srt` (subtitles), `.md` (readable paragraphs, best input for study notes).
A video counts as finished only after all three are written.

## Step 1: Install (PyAV pinned to 18.1.0)

In [ ]:
!pip install -q -U faster-whisper "av==18.1.0"

import sys
from importlib.metadata import version
av_v = version("av")
print("faster-whisper:", version("faster-whisper"), "| av:", av_v)
if av_v != "18.1.0":
    print("\n>>> av is not 18.1.0. Runtime -> Restart session, then run this cell again.")
elif "av" in sys.modules:
    import av
    if av.__version__ != "18.1.0":
        print("\n>>> An older av is still loaded. Runtime -> Restart session, then run this cell again.")
    else:
        print("av OK")
else:
    print("av OK")

!nvidia-smi --query-gpu=name,memory.total --format=csv

## Step 1b: CUDA 12 libraries

In [ ]:
!pip install -q nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"

import ctypes, glob, os
import nvidia.cublas.lib, nvidia.cudnn.lib

libs = []
for pkg in (nvidia.cublas.lib, nvidia.cudnn.lib):
    d = list(pkg.__path__)[0]
    libs += sorted(glob.glob(os.path.join(d, "*.so*")))

for _ in range(2):          # two passes so dependent libraries load in a working order
    for so in libs:
        try:
            ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
        except OSError:
            pass

ctypes.CDLL("libcublas.so.12")
print("CUDA 12 libraries loaded OK")

## Step 2: Token, Drive and settings

In [ ]:
import os, logging

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF token loaded from Colab secrets.")
except Exception:
    print("No HF_TOKEN secret found. Continuing without it.")
    logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# ---------------- SETTINGS ----------------
BASE        = Path("/content/drive/MyDrive/transcribe")
AUDIO_DIR   = BASE / "audio"
OUT_DIR     = BASE / "transcripts_large_v3"
MODEL_NAME  = "large-v3"
LANGUAGE    = "en"

SESSION_RANGE = None   # None = all sessions. Or (1, 20) to only handle Session 1 to Session 20
FILE_LIMIT  = None     # None = process everything left. Or a number, e.g. 25, to do a batch
MAX_HOURS   = 10       # stop STARTING new files after this many hours (a clean stop before the session limit). None = no limit

USE_BATCHED = False    # False = the same method you already tested (safe).
                       # True  = faster batched mode (often 2-4x faster on a good GPU). Test it on 1 file first.
BATCH_SIZE  = 16       # only used if USE_BATCHED = True. Lower to 8 if you see an out-of-memory error.

PARA_SECONDS = 45      # length of each paragraph in the .md file

HOTWORDS    = ("Naresh IT, Veera Babu, Kubernetes, kubectl, Terraform, AWS, EC2, S3, IAM, VPC, "
               "Docker, Jenkins, Helm, CI/CD, GitHub Actions, Ansible, Prometheus, Grafana, "
               "Azure, GCP, YAML, DevOps, SRE")
# ------------------------------------------

OUT_DIR.mkdir(parents=True, exist_ok=True)
exts = {".mp3", ".m4a", ".wav", ".mp4", ".mkv", ".webm", ".aac", ".flac", ".ogg"}
import re
def session_num(p):
    m = re.search(r"\d+", p.stem)          # first number in the file name
    return int(m.group()) if m else None

def natural_key(p):
    n = session_num(p)
    return (n if n is not None else 10**9, p.name)

files = sorted((p for p in AUDIO_DIR.iterdir() if p.suffix.lower() in exts), key=natural_key)

if SESSION_RANGE:
    lo, hi = SESSION_RANGE
    files = [p for p in files if session_num(p) is not None and lo <= session_num(p) <= hi]

no_number = [p.name for p in files if session_num(p) is None]
if no_number:
    print("Warning: no number found in these names (they go last):", no_number)
done = [p for p in files if (OUT_DIR / f"{p.stem}.txt").exists()]
print(f"Audio files: {len(files)} | already done: {len(done)} | remaining: {len(files) - len(done)}")
print("Order check (first 5):", [p.name for p in files if (OUT_DIR / f"{p.stem}.txt").exists() is False][:5])

## Step 3: Transcribe everything (resumable)
Keep this browser tab open while it runs. If it stops, run Steps 1, 1b, 2 and 3 again.

In [ ]:
import time, shutil
from faster_whisper import WhisperModel, BatchedInferencePipeline

model = WhisperModel(MODEL_NAME, device="cuda", compute_type="float16")
batched = BatchedInferencePipeline(model=model) if USE_BATCHED else None

def ts(sec, srt=False):
    h, rem = divmod(int(sec), 3600)
    m, s = divmod(rem, 60)
    if srt:
        ms = int((sec - int(sec)) * 1000)
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"
    return f"{h:02d}:{m:02d}:{s:02d}"

def run_transcribe(path):
    if USE_BATCHED:
        return batched.transcribe(
            path, language=LANGUAGE, batch_size=BATCH_SIZE,
            beam_size=5, initial_prompt=HOTWORDS,
        )
    return model.transcribe(
        path, language=LANGUAGE, vad_filter=True, beam_size=5,
        condition_on_previous_text=False,     # avoids repeated-text loops on long audio
        initial_prompt=HOTWORDS,
    )

todo = [f for f in files if not (OUT_DIR / f"{f.stem}.txt").exists()]
if FILE_LIMIT:
    todo = todo[:FILE_LIMIT]
print(f"{len(todo)} file(s) to process | model={MODEL_NAME} | batched={USE_BATCHED}\n")

T0 = time.time()
ok, failed, audio_min_total = 0, [], 0.0

for n, f in enumerate(todo, 1):
    if MAX_HOURS and (time.time() - T0) > MAX_HOURS * 3600:
        print(f"\nReached MAX_HOURS={MAX_HOURS}. Stopping cleanly. Run Step 3 again to continue.")
        break

    t0 = time.time()
    local = Path("/content") / f.name
    try:
        shutil.copy(f, local)                 # read from fast local disk instead of Drive
        segments, info = run_transcribe(str(local))

        txt_tmp  = OUT_DIR / f"{f.stem}.txt.part"
        srt_tmp  = OUT_DIR / f"{f.stem}.srt.part"
        md_tmp   = OUT_DIR / f"{f.stem}.md.part"
        paras = []                            # each item: [start_seconds, [texts]]

        with open(txt_tmp, "w", encoding="utf-8") as ft, open(srt_tmp, "w", encoding="utf-8") as fs:
            for i, s in enumerate(segments, 1):
                text = s.text.strip()
                ft.write(f"[{ts(s.start)}] {text}\n")
                fs.write(f"{i}\n{ts(s.start, True)} --> {ts(s.end, True)}\n{text}\n\n")
                if not paras or s.start - paras[-1][0] >= PARA_SECONDS:
                    paras.append([s.start, []])
                paras[-1][1].append(text)

        with open(md_tmp, "w", encoding="utf-8") as fm:
            fm.write(f"# {f.stem}\n\n")
            for start, texts in paras:
                fm.write(f"**[{ts(start)}]** " + " ".join(texts) + "\n\n")

        srt_tmp.rename(OUT_DIR / f"{f.stem}.srt")
        md_tmp.rename(OUT_DIR / f"{f.stem}.md")
        txt_tmp.rename(OUT_DIR / f"{f.stem}.txt")   # .txt is renamed last: it marks the file as finished

        took = time.time() - t0
        dur = info.duration
        ok += 1
        audio_min_total += dur / 60
        avg_took = (time.time() - T0) / ok
        left = len(todo) - n
        print(f"[{n}/{len(todo)}] {f.name} | {dur/60:.0f} min audio | {took/60:.1f} min | {dur/took:.0f}x"
              f" | about {left*avg_took/3600:.1f} h left")
    except Exception as e:
        failed.append(f.name)
        print(f"[{n}/{len(todo)}] FAILED: {f.name} -> {type(e).__name__}: {e}")
    finally:
        local.unlink(missing_ok=True)

remaining = len([f for f in files if not (OUT_DIR / f"{f.stem}.txt").exists()])
print("\n---------- SUMMARY ----------")
print(f"Finished this run : {ok} file(s), {audio_min_total/60:.1f} h of audio in {(time.time()-T0)/3600:.2f} h")
print(f"Failed            : {len(failed)} {failed if failed else ''}")
print(f"Still remaining   : {remaining}")
if remaining:
    print("Run Step 3 again to continue (failed files are retried).")
else:
    print("All files are transcribed.")

## Step 4: Spot-check one result
Prints the first lines of the most recent transcript. Also open one `.md` file from `transcripts_large_v3/` and skim the middle.

In [ ]:
latest = max(OUT_DIR.glob("*.txt"), key=lambda p: p.stat().st_mtime, default=None)
if latest:
    print(latest.name, "\n")
    print("".join(open(latest, encoding="utf-8").readlines()[:20]))
else:
    print("No transcripts yet.")